# The Bridge and the Torch

In the darkness of the night, a group of four individuals encounters a river. A slender bridge stretches before them, capable of accommodating just two people simultaneously. Equipped with a single torch, they must rely on its flickering light to navigate the bridge. Each person possesses a distinct crossing time: Ariela takes 1 minute, Brian takes 2 minutes, Charly takes 5 minutes, and Dumpy takes 8 minutes. It is crucial to note that when two people cross together, they must synchronize their steps with the slower individual's pace. Given the torch's limited lifespan of 15 minutes, the pressing question arises: can all four individuals successfully traverse the bridge?

We load the Java API of `Z3` from *Maven Central* and import the package `com.microsoft.z3`.  All `Z3` objects are created by a *context*, which we call `Z3`.

In [ ]:
%maven tools.aqua:z3-turnkey:4.14.1
import com.microsoft.z3.*;

In [ ]:
Context Z3 = new Context();

We need the following variables to encode the problem:
* `A` equals `1` if Ariela is on the left shore,
* `B` equals `1` if Brian  is on the left shore,
* `C` equals `1` if Charly is on the left shore,
* `D` equals `1` if Dumpy  is on the left shore,
* `T` equals the time that has passed.

`start` returns a set of constraints that specify that everybody is on the left side of the river.
Furthermore, no time has passed.
$$ \texttt{A} = 1 \wedge \texttt{B} = 1 \wedge \texttt{C} = 1 \wedge \texttt{D} = 1 \wedge \texttt{T} = 0 $$

In [ ]:
List<BoolExpr> start(IntExpr A, IntExpr B, IntExpr C, IntExpr D, IntExpr T) {
    return List.of();  // your code here
}

`goal` returns a set of constraints that specify that everybody is on the right side of the river.
The time must be less or equal than $15$:
$$ \texttt{A} = 0 \wedge \texttt{B} = 0 \wedge \texttt{C} = 0 \wedge \texttt{D} = 0 \wedge \texttt{T} \leq 15 $$

In [ ]:
List<BoolExpr> goal(IntExpr A, IntExpr B, IntExpr C, IntExpr D, IntExpr T) {
    return List.of();  // your code here
}

`torchOK` descrives one crossing of the bridge from left to right.
* `A` is the number of Arielas on the left side before the crossing, 
   while `Ax` is the number of Arielas on the left side after the crossing.
* Dito for `B`, `Bx`, $\cdots$, `D`, `Dx`.
* `T` is the time before the crossing and `Tx` is the time after the crossing.

The constraints ensure that
 * People go from left to right,
 * at least one person crosses the bridge,
 * at most two persons cross the bridge,
 * the variable `Tx` is incremented by the right amount.

In [ ]:
List<BoolExpr> torchOK(IntExpr A,  IntExpr B,  IntExpr C,  IntExpr D,  IntExpr T,
                       IntExpr Ax, IntExpr Bx, IntExpr Cx, IntExpr Dx, IntExpr Tx) {
    return List.of();  // your code here
}

`transition` returns a set of constraints that describe the crossings of the bridge.
The variable `i` specifies the number of the crossing.  The first crossing has `i == 0`.

In [ ]:
List<BoolExpr> transition(IntExpr A,  IntExpr B,  IntExpr C,  IntExpr D,  IntExpr T,
                          IntExpr Ax, IntExpr Bx, IntExpr Cx, IntExpr Dx, IntExpr Tx, int i) {
    if (i % 2 == 0) {
        return List.of();  // your code here
    } else {
        return List.of();  // your code here
    }
}

`bridgeCSP` tries to solve the problem using `n` crossings of the bridge.
If this is possible, a solution is returned that is a map mapping the variables
to their values.  Otherwise, `null` is returned.

The helper function `addAll` adds a list of constraints to a solver.

In [ ]:
void addAll(Solver solver, List<BoolExpr> constraints) {
    solver.add(constraints.toArray(BoolExpr[]::new));
}

In [ ]:
Map<String, Integer> bridgeCSP(int n) {
    Solver solver = Z3.mkSolver();
    List<IntExpr> As = new ArrayList<>(), Bs = new ArrayList<>(), Cs = new ArrayList<>(),
                  Ds = new ArrayList<>(), Ts = new ArrayList<>();
    for (int i = 0; i <= n; i++) {
        As.add(Z3.mkIntConst("A" + i));
        Bs.add(Z3.mkIntConst("B" + i));
        Cs.add(Z3.mkIntConst("C" + i));
        Ds.add(Z3.mkIntConst("D" + i));
        Ts.add(Z3.mkIntConst("T" + i));
    }
    addAll(solver, start(As.get(0), Bs.get(0), Cs.get(0), Ds.get(0), Ts.get(0)));
    addAll(solver, goal (As.get(n), Bs.get(n), Cs.get(n), Ds.get(n), Ts.get(n)));
    for (int i = 0; i < n; i++) {
        addAll(solver, transition(As.get(i),     Bs.get(i),     Cs.get(i),     Ds.get(i),     Ts.get(i),
                                  As.get(i + 1), Bs.get(i + 1), Cs.get(i + 1), Ds.get(i + 1), Ts.get(i + 1), i));
    }
    // Bounds
    for (int i = 0; i <= n; i++) {
        for (var X : List.of(As, Bs, Cs, Ds)) {
            solver.add(Z3.mkGe(X.get(i), Z3.mkInt(0)), Z3.mkLe(X.get(i), Z3.mkInt(1)));
        }
        solver.add(Z3.mkGe(Ts.get(i), Z3.mkInt(0)));
    }
    if (solver.check() == Status.SATISFIABLE) {
        Model model = solver.getModel();
        Map<String, Integer> solution = new LinkedHashMap<>();
        for (int i = 0; i <= n; i++) {
            solution.put("A" + i, Integer.parseInt(model.eval(As.get(i), true).toString()));
            solution.put("B" + i, Integer.parseInt(model.eval(Bs.get(i), true).toString()));
            solution.put("C" + i, Integer.parseInt(model.eval(Cs.get(i), true).toString()));
            solution.put("D" + i, Integer.parseInt(model.eval(Ds.get(i), true).toString()));
            solution.put("T" + i, Integer.parseInt(model.eval(Ts.get(i), true).toString()));
        }
        return solution;
    }
    return null;
}

The function `findSolution` tries to solve the problem with an increasing number of crossings.  It returns a pair consisting of the number `n` of crossings and the solution.

In [ ]:
record Result(int n, Map<String, Integer> solution) {}

Result findSolution() {
    int n = 1;
    while (true) {
        System.out.println(n);
        var solution = bridgeCSP(n);
        if (solution != null) {
            return new Result(n, solution);
        }
        n += 2;
    }
}

In [ ]:
long start = System.currentTimeMillis();
var result = findSolution();
System.out.println("Bridge Solving: " + (System.currentTimeMillis() - start) + " ms");
result

In [ ]:
void showSolution(Map<String, Integer> sol, int n) {
    for (int i = 0; i <= n; i++) {
        int A = sol.get("A" + i);
        int B = sol.get("B" + i);
        int C = sol.get("C" + i);
        int D = sol.get("D" + i);
        int T = sol.get("T" + i);
        String left  = "🏃‍♀️".repeat(A) + "🏃🏽‍♂️".repeat(B) + "🚶🏽‍♂️".repeat(C) + "👨‍🦽".repeat(D);
        String right = "🏃‍♀️".repeat(1 - A) + "🏃🏽‍♂️".repeat(1 - B) + "🚶🏽‍♂️".repeat(1 - C) + "👨‍🦽".repeat(1 - D);
        System.out.println(String.format("%-20s", left) + " ".repeat(22) + right);
        System.out.println("🕰️" + T);
        if (i % 2 == 0 && i < n) {
            int PS = sol.get("A" + i) - sol.get("A" + (i + 1));
            int BS = sol.get("B" + i) - sol.get("B" + (i + 1));
            int CS = sol.get("C" + i) - sol.get("C" + (i + 1));
            int FS = sol.get("D" + i) - sol.get("D" + (i + 1));
            String cross = "🏃‍♀️".repeat(PS) + "🏃🏽‍♂️".repeat(BS) + "🚶🏽‍♂️".repeat(CS) + "👨‍🦽".repeat(FS);
            System.out.println(" ".repeat(24) + ">> " + cross + " >>");
        } else if (i + 1 <= n) {
            int PS = sol.get("A" + (i + 1)) - sol.get("A" + i);
            int BS = sol.get("B" + (i + 1)) - sol.get("B" + i);
            int CS = sol.get("C" + (i + 1)) - sol.get("C" + i);
            int FS = sol.get("D" + (i + 1)) - sol.get("D" + i);
            String cross = "🏃‍♀️".repeat(PS) + "🏃🏽‍♂️".repeat(BS) + "🚶🏽‍♂️".repeat(CS) + "👨‍🦽".repeat(FS);
            System.out.println(" ".repeat(24) + "<< " + cross + " <<");
        }
    }
}

In [ ]:
showSolution(result.solution(), result.n());

The function `showAnimatedSolution` displays an animation of the solution.  The animation is implemented in *JavaScript*, which is executed by the browser.  The Java code only inserts the data describing the solution into the HTML code: The placeholders `@uid@`, `@n@`, and `@states@` are replaced by a unique identifier, the number of crossings, and the list of states, respectively.  The list of states is given in *JSON* notation.

In [ ]:
void showAnimatedSolution(Map<String, Integer> sol, int n) {
    if (sol == null) {
        return;
    }
    var states = new StringJoiner(", ", "[", "]");
    for (int i = 0; i <= n; i++) {
        states.add("{\"A\": %d, \"B\": %d, \"C\": %d, \"D\": %d, \"T\": %d, \"torchLeft\": %b}".formatted(
            sol.get("A" + i), sol.get("B" + i), sol.get("C" + i), sol.get("D" + i), sol.get("T" + i), i % 2 == 0));
    }
    String uid = Integer.toString(new Random().nextInt(Integer.MAX_VALUE), 36);
    String htmlContent = """
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-@uid@" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-@uid@" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-@uid@" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-@uid@" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of @n@</span>
        </div>
        <div style="margin-bottom: 10px; font-size: 18px; font-weight: bold; color: #d32f2f;">Time Elapsed: <span id="timer-@uid@">0</span> mins</div>
        <svg width="600" height="300" style="background-color: #0b1120; border: 1px solid #333; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="300" fill="#1e3a29" />
            <rect x="450" y="0" width="150" height="300" fill="#1e3a29" />
            <rect x="150" y="130" width="300" height="40" fill="#5c4033" stroke="#3b261e" stroke-width="4" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#a5d6a7" text-anchor="middle">Left Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#a5d6a7" text-anchor="middle">Right Shore</text>
            
            <text id="west-a-@uid@" x="75" y="100" font-size="24" text-anchor="middle"></text>
            <text id="west-b-@uid@" x="75" y="140" font-size="24" text-anchor="middle"></text>
            <text id="west-c-@uid@" x="75" y="180" font-size="24" text-anchor="middle"></text>
            <text id="west-d-@uid@" x="75" y="220" font-size="24" text-anchor="middle"></text>

            <text id="east-a-@uid@" x="525" y="100" font-size="24" text-anchor="middle"></text>
            <text id="east-b-@uid@" x="525" y="140" font-size="24" text-anchor="middle"></text>
            <text id="east-c-@uid@" x="525" y="180" font-size="24" text-anchor="middle"></text>
            <text id="east-d-@uid@" x="525" y="220" font-size="24" text-anchor="middle"></text>

            <g id="group-@uid@" style="transition: transform 1.2s ease-in-out;">
                <text x="200" y="110" font-size="28" text-anchor="middle">🔦</text>
                <text id="bridge-p-@uid@" x="200" y="160" font-size="24" text-anchor="middle"></text>
            </g>
        </svg>
    </div>
    <script>
    (function() {
        const states = @states@;
        let currentStep = 0;
        let isAnimating = false;
        
        const group = document.getElementById('group-@uid@');
        const wA = document.getElementById('west-a-@uid@');
        const wB = document.getElementById('west-b-@uid@');
        const wC = document.getElementById('west-c-@uid@');
        const wD = document.getElementById('west-d-@uid@');
        const eA = document.getElementById('east-a-@uid@');
        const eB = document.getElementById('east-b-@uid@');
        const eC = document.getElementById('east-c-@uid@');
        const eD = document.getElementById('east-d-@uid@');
        const bridgeP = document.getElementById('bridge-p-@uid@');
        const stepLabel = document.getElementById('label-step-@uid@');
        const timerLabel = document.getElementById('timer-@uid@');
        
        function renderStaticState(step) {
            const s = states[step];
            wA.textContent = '🏃‍♀️'.repeat(s.A);
            wB.textContent = '🏃🏽‍♂️'.repeat(s.B);
            wC.textContent = '🚶🏽‍♂️'.repeat(s.C);
            wD.textContent = '👨‍🦽'.repeat(s.D);
            eA.textContent = '🏃‍♀️'.repeat(1 - s.A);
            eB.textContent = '🏃🏽‍♂️'.repeat(1 - s.B);
            eC.textContent = '🚶🏽‍♂️'.repeat(1 - s.C);
            eD.textContent = '👨‍🦽'.repeat(1 - s.D);
            bridgeP.textContent = '';
            
            group.style.transform = s.torchLeft ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of @n@';
            timerLabel.textContent = s.T;
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const aPass = Math.abs(s0.A - s1.A);
            const bPass = Math.abs(s0.B - s1.B);
            const cPass = Math.abs(s0.C - s1.C);
            const dPass = Math.abs(s0.D - s1.D);
            
            if (s0.torchLeft) { 
                wA.textContent = '🏃‍♀️'.repeat(s0.A - aPass);
                wB.textContent = '🏃🏽‍♂️'.repeat(s0.B - bPass);
                wC.textContent = '🚶🏽‍♂️'.repeat(s0.C - cPass);
                wD.textContent = '👨‍🦽'.repeat(s0.D - dPass);
            } else { 
                eA.textContent = '🏃‍♀️'.repeat((1 - s0.A) - aPass);
                eB.textContent = '🏃🏽‍♂️'.repeat((1 - s0.B) - bPass);
                eC.textContent = '🚶🏽‍♂️'.repeat((1 - s0.C) - cPass);
                eD.textContent = '👨‍🦽'.repeat((1 - s0.D) - dPass);
            }
            bridgeP.textContent = '🏃‍♀️'.repeat(aPass) + '🏃🏽‍♂️'.repeat(bPass) + '🚶🏽‍♂️'.repeat(cPass) + '👨‍🦽'.repeat(dPass);
            
            group.style.transform = s1.torchLeft ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1200));
            
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-@uid@').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });
        document.getElementById('btn-prev-@uid@').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });
        document.getElementById('btn-reset-@uid@').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    
    """.replace("@uid@", uid)
       .replace("@n@", String.valueOf(n))
       .replace("@states@", states.toString());
    display(htmlContent, "text/html");
}

In [ ]:
showAnimatedSolution(result.solution(), result.n());